## cPCE for real-valued optimization (continuous benchmark functions)

Continuous Pauli Correlation Encoding (cPCE) reuses the same measurement machinery as `PCE`, but instead of squashing each Pauli correlator's expectation value through `tanh` and thresholding it with the sign function to obtain a discrete bit, it uses the expectation value itself as a real-valued decision variable. This makes it suitable for real-valued (rather than binary) optimization problems.

This implementation follows the approach described in https://arxiv.org/abs/2604.05637.

Here we benchmark `cPCE` on a shifted [Rosenbrock function](https://en.wikipedia.org/wiki/Rosenbrock_function) (the classic "banana function"), a standard test problem for real-valued optimizers.

In [ ]:
from qarp.algorithms import cPCE
from qarp.algorithms import calculate_qubits
from qarp.blocks import HEABlock
from qarp.algorithms import StateVector
from qarp.optimizers import ScipyOptimizer
from qarp.graphs import Graph

import numpy as np
import matplotlib.pyplot as plt

from qarp import config

config.seed = 1234
np.random.seed(config.seed)

### The benchmark: a shifted Rosenbrock function

The Rosenbrock function is $f(z) = \sum_i \left[100 (z_{i+1} - z_i^2)^2 + (1 - z_i)^2\right]$, minimized at $z_i = 1$ for every $i$.

Every correlator's expectation value naturally lives in $[-1, 1]$, so on its own it cannot represent decision variables outside that range. To make the point clearly, we *shift* the problem so that its optimum ($y_i = \mathrm{SHIFT} + 1 = 3$) sits outside $[-1, 1]$. `cPCE` exposes a `coefficients` argument for exactly this: it multiplies each expectation value before it is handed to `classical_function`/`quantum_function`, rescaling $[-1, 1]$ onto whatever range the real variables of the problem need. We compare running `cPCE` with and without it.

In [ ]:
SHIFT = 2.0


def rosenbrock(z):
    return sum(100.0 * (z[i + 1] - z[i] ** 2) ** 2 + (1.0 - z[i]) ** 2 for i in range(len(z) - 1))


def rosenbrock_classical_function(graph, values, coefficients=None):
    """classical_function: scores a real-valued solution on the shifted Rosenbrock function."""
    if coefficients is not None:
        values = [v * c for v, c in zip(values, coefficients, strict=True)]
    z = [v - SHIFT for v in values]
    return rosenbrock(z)


def rosenbrock_quantum_function(result, graph, n_qubits, order, coefficients=None):
    """quantum_function: energy to minimize, built from the raw Pauli expectation values."""
    solution = np.array(result, dtype=float)
    if coefficients is not None:
        solution = solution * np.array(coefficients, dtype=float)
    z = solution - SHIFT
    loss = rosenbrock(z)
    return loss, 0.0, list(solution)

### Ansatz and problem size

`cPCE` reuses the same qubit-sizing logic as `PCE`: `calculate_qubits` gives the minimum number of qubits needed to fit `n_vars` real-valued variables at a given correlator `order`. The graph passed to `cPCE` only needs to declare the nodes (one per decision variable); no edges are required since our custom functions don't use graph connectivity.

In [ ]:
n_vars = 4
order = 2

graph = Graph()
graph.add_nodes_from(range(n_vars))

n_qubits = calculate_qubits(n_vars, order)
print("Number of qubits:", n_qubits)

he_wfn = HEABlock(n_qubits, 3, True, True, True, False).build()

### Without `coefficients`

Expectation values stay bounded in $[-1, 1]$, so the shifted optimum at $y_i = 3$ is unreachable. We expect the optimizer to get stuck near the boundary of the accessible range.

In [ ]:
cpce_unscaled = cPCE(
    graph=graph,
    order=order,
    ket=he_wfn,
    classical_function=rosenbrock_classical_function,
    quantum_function=rosenbrock_quantum_function,
    coefficients=None,
    primitive=StateVector(),
    initial_parameters=[0.1] * len(list(he_wfn.symbols)),
    verbose=False,
    optimizer=ScipyOptimizer("COBYQA"),
).build()

_, x, solution_unscaled = cpce_unscaled.run()

print("Solution (unscaled):", solution_unscaled)
print("f(y):", rosenbrock_classical_function(graph, solution_unscaled))

### With `coefficients`

Scaling every expectation value by `coefficients=[10.0] * n_vars` rescales $[-1, 1]$ onto $[-10, 10]$, which comfortably contains the shifted optimum $y_i = 3$.

In [ ]:
coefficients = [10.0] * n_vars

cpce_scaled = cPCE(
    graph=graph,
    order=order,
    ket=he_wfn,
    classical_function=rosenbrock_classical_function,
    quantum_function=rosenbrock_quantum_function,
    coefficients=coefficients,
    primitive=StateVector(),
    initial_parameters=[0.1] * len(list(he_wfn.symbols)),
    verbose=False,
    optimizer=ScipyOptimizer("COBYQA"),
).build()

_, x, solution_scaled = cpce_scaled.run()

print("Solution (scaled):", solution_scaled)
print("f(y):", rosenbrock_classical_function(graph, solution_scaled))

### Convergence

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot([h[1] for h in cpce_unscaled.history], label="Unscaled")
plt.ylabel("Rosenbrock f(y)")
plt.xlabel("Iteration")
plt.yscale("log")
plt.grid()
plt.plot([h[1] for h in cpce_scaled.history], label="Scaled (x10.0)")
plt.ylabel("Rosenbrock f(y)")
plt.xlabel("Iteration")
plt.yscale("log")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

As expected, without `coefficients` the solution cannot leave $[-1, 1]$ and the loss plateaus far from zero, while rescaling with `coefficients` lets `cPCE` reach the neighborhood of the true optimum $y_i = 3$, driving the Rosenbrock loss much closer to its global minimum of 0.